# Homework 11: querying a materials database

Covers Lecture 29 (querying Materials Project in a loop), Lecture 30 (composition as a feature), Lecture 31 (missing values and provenance), and Lecture 32 (stability and the convex hull).

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

The `ipynb` file type is the only accepted format.

Save a copy before beginning.
Reloading or disconnecting can revert the notebook to the instructor copy.

# Problem 0 (-10 pts if blank): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work. Critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

# Setup

Run these two cells first.
They need the `MaterialsProject` secret from the API-key checklist on Canvas.

> The install takes a minute or two. If Colab asks you to restart the session, restart and run both cells again.

In [ ]:
!pip install mp-api

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata
from mp_api.client import MPRester

API_KEY = userdata.get('MaterialsProject')     # your key, read from Colab Secrets

# Problem 1 (30 pts): querying a database in a loop

The Period 4 transition metals run from scandium (Sc) to zinc (Zn).
Materials Project lists a computed density for the stable crystal of each one.

(a) Make a list `metals` holding the ten chemical symbols from Sc to Zn, in order.

- Print the list.

In [ ]:
# your code here

(b) Loop over `metals` and query each one with `mpr.materials.summary.search`.

- Pass `chemsys=symbol`, `is_stable=True`, and `num_elements=1`.
- Request the fields `material_id` and `density`.
- Append one dictionary per metal to a list called `records`.
- Give each dictionary the keys `'symbol'`, `'material_id'`, and `'density'`.

> Open the `MPRester` once, outside the loop, as in lecture.
> Each search returns a list. Use its first document, `docs[0]`.

In [ ]:
# your code here

(c) Turn `records` into a `DataFrame` called `metal_table`.

- Print `metal_table`.

In [ ]:
# your code here

(d) Plot the density of each metal as a bar chart, in the order of `metals`.

- Label the y axis with its units, g/cm³.

In [ ]:
# your code here

(e) These are measured densities at room temperature (*CRC Handbook of Chemistry and Physics*):

| metal | measured density (g/cm³) |
|---|---|
| Ti | 4.51 |
| Fe | 7.87 |
| Cu | 8.96 |

- Print the Materials Project density next to the measured one for each of the three.
- Say what each of the two densities describes.
- Give one reason they can differ.

In [ ]:
# your code here

*your answer here*

# Problem 2 (25 pts): composition as a feature

Iron and oxygen form several compounds (FeO, Fe₃O₄, Fe₂O₃, and others), each in one or more crystal structures.
Problems 2 to 4 use every Fe-O entry in the database, stable or not.

(a) Query the summary endpoint with `chemsys='Fe-O'` and no stability filter.

- Request `material_id`, `formula_pretty`, `composition`, `formation_energy_per_atom`, `energy_above_hull`, `is_stable`, `theoretical`, and `bulk_modulus`.
- Print how many documents came back.

In [ ]:
# your code here

(b) Loop over the documents and build a list of dictionaries, one per entry.

- Keep `material_id`, `formula_pretty`, `formation_energy_per_atom`, `energy_above_hull`, `is_stable`, and `theoretical`.
- Add the key `'x_O'`, the atomic fraction of oxygen from the entry's `composition`.
- Turn the list into a `DataFrame` called `fe_o`.
- Print its first five rows.

> `bulk_modulus` is handled in Problem 4. Leave it out here.

In [ ]:
# your code here

(c) Plot formation energy per atom against `x_O` as a scatter plot.

- Label both axes.

> Formation energy is in eV/atom. `x_O` has no units.

In [ ]:
# your code here

# Problem 3 (30 pts): stability and the convex hull

(a) Select the rows of `fe_o` whose `energy_above_hull` is 0.

- Print their formulas and `x_O`.
- Print how many rows have `is_stable` equal to `True`.

In [ ]:
# your code here

(b) Sort the stable rows by `x_O`.
Build two lists for the lower hull, `hull_x` for `x_O` and `hull_e` for formation energy.

- Start both lists with pure Fe, at `x_O` = 0.
- Append the sorted stable rows in a loop.
- End both lists with pure O, at `x_O` = 1.
- Print `hull_x` and `hull_e`.

> A pure element has a formation energy of 0 by definition. Both end points sit at 0 eV/atom.

In [ ]:
# your code here

(c) Remake the scatter plot from Problem 2(c).
Join the hull points with a line on the same chart.

- Add a legend naming the points and the line.

In [ ]:
# your code here

(d) Choose one unstable entry whose formula matches a stable one.

- Print its `material_id`, formula, and `energy_above_hull`.
- Say what that energy above the hull tells you about this entry.

In [ ]:
# your code here

*your answer here*

# Problem 4 (15 pts): missing values and provenance

Only some entries have a computed bulk modulus.
When one is missing, `doc.bulk_modulus` is `None` instead of a dictionary.

Run this cell to add a `bulk_modulus` column to `fe_o`.

> Change `docs` to the name of your list from Problem 2(a) if it differs.

In [ ]:
k_values = []
for doc in docs:
    try:
        k_values.append(doc.bulk_modulus['vrh'])   # GPa, Voigt-Reuss-Hill average
    except TypeError:                              # bulk_modulus is None: no elastic calculation
        k_values.append(np.nan)
fe_o['bulk_modulus'] = k_values

(a) Count the missing values in the `bulk_modulus` column.

- Print the number of entries with a bulk modulus.
- Print the number without one.

In [ ]:
# your code here

(b) The `theoretical` field is `True` for entries with no matching experimentally reported structure.

- Print how many `fe_o` entries are theoretical.
- Say what `theoretical` means for using one of those entries as evidence about real iron oxides.

In [ ]:
# your code here

*your answer here*

(c) Look up how Materials Project asks to be cited, on its website.

- Write the citation you would use for the data in this assignment, including the database version.

*your answer here*